# 🎬 ViralCut AI — by Kamran AI (Google Colab Edition)

> **Local-First AI Video Clipping, Reframing & Styled Captioning in Google Colab**

ViralCut AI transforms YouTube videos or uploaded files into short, reframed, styled-captioned clips (9:16 Shorts/Reels) — running **100% locally** inside your Google Colab instance.

### ✨ What This Notebook Does:
1. **Clones Repository**: Downloads the public [ViralCut AI](https://github.com/itxunknown39-web/ViralCut-AI.git) codebase.
2. **Environment & Hardware Inspection**: Probes CPU, RAM, CUDA GPU, and disk space.
3. **Installs Dependencies**: Sets up FFmpeg, PyTorch, faster-whisper, yt-dlp, and web application dependencies.
4. **Model Preparation**: Caches `faster-whisper` speech recognition weights locally.
5. **Launches Server**: Runs the FastAPI application backend and React dashboard frontend.
6. **Exposes Public Tunnel**: Uses Cloudflare Quick Tunnel (`trycloudflare.com`) to generate a free, secure public URL.

🔒 **Zero API Keys Required**: No OpenAI, Anthropic, Cloudflare, or paid API tokens needed.

In [ ]:
# ==========================================
# 2. REPOSITORY CLONING
# ==========================================
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/itxunknown39-web/ViralCut-AI.git"
REPO_DIR = Path("/content/ViralCut-AI") if os.path.exists("/content") else Path.cwd() / "ViralCut-AI"

if REPO_DIR.exists() and (REPO_DIR / "app").exists():
    print(f"📁 Repository directory '{REPO_DIR.name}' already exists. Fetching latest updates...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)
else:
    print(f"📦 Cloning repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"✅ Active working directory: {Path.cwd()}")

In [ ]:
# ==========================================
# 3. ENVIRONMENT & HARDWARE DIAGNOSTICS
# ==========================================
import platform
import psutil
import shutil

print("==================================================")
print("🔍 SYSTEM DIAGNOSTICS REPORT")
print("==================================================")
print(f"• Python Version: {platform.python_version()}")
print(f"• Operating System: {platform.system()} {platform.release()}")

ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"• Total System RAM: {ram_gb:.2f} GB")

total_disk, used_disk, free_disk = shutil.disk_usage("/")
print(f"• Disk Space Free: {free_disk / (1024 ** 3):.2f} GB / {total_disk / (1024 ** 3):.2f} GB")

# Probe CUDA GPU via nvidia-smi
try:
    smi_res = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True)
    if smi_res.returncode == 0 and smi_res.stdout.strip():
        parts = [p.strip() for p in smi_res.stdout.strip().split(",")]
        print(f"🚀 GPU Detected: {parts[0]} ({parts[1]} VRAM, Driver: {parts[2]})")
    else:
        print("⚠️ GPU Info: No NVIDIA GPU detected via nvidia-smi (CPU execution mode)")
except Exception:
    print("⚠️ GPU Info: nvidia-smi probe unavailable")

print("==================================================")

In [ ]:
# ==========================================
# 4. FFMPEG INSTALLATION & VERIFICATION
# ==========================================
def check_ffmpeg():
    try:
        res = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
        return res.returncode == 0
    except Exception:
        return False

if check_ffmpeg():
    print("✅ FFmpeg is installed and accessible on PATH.")
else:
    print("📦 Installing FFmpeg via system package manager...")
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
    if check_ffmpeg():
        print("✅ FFmpeg successfully installed.")
    else:
        print("❌ Error installing FFmpeg.")

In [ ]:
# ==========================================
# 5. PYTHON DEPENDENCY INSTALLATION
# ==========================================
import sys

print("📦 Upgrading yt-dlp to latest release...")
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "-q", "yt-dlp"], check=False)

req_file = REPO_DIR / "requirements.txt"
if req_file.exists():
    print(f"📦 Installing dependencies from {req_file.name}...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)], check=True)
    print("✅ All Python dependencies installed successfully.")
else:
    print("⚠️ requirements.txt not found. Installing base packages...")
    packages = ["fastapi", "uvicorn[standard]", "python-multipart", "pydantic>=2.0", "faster-whisper", "requests", "indic-transliteration"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

In [ ]:
# ==========================================
# 6. GPU & COMPUTE ENGINE CONFIGURATION
# ==========================================
cuda_active = False
gpu_name_str = "CPU"

try:
    import torch
    cuda_active = torch.cuda.is_available()
    if cuda_active:
        gpu_name_str = torch.cuda.get_device_name(0)
except Exception:
    try:
        from ctranslate2 import get_cuda_device_count
        cuda_active = get_cuda_device_count() > 0
        if cuda_active:
            gpu_name_str = "NVIDIA CUDA GPU"
    except Exception:
        cuda_active = False

print("==================================================")
if cuda_active:
    print(f"🚀 GPU DETECTED → CUDA Acceleration Active ({gpu_name_str})")
    print("   Whisper & FFmpeg will utilize GPU acceleration for maximum performance.")
else:
    print("⚠️ GPU UNAVAILABLE → CPU Execution Mode Active")
    print("   Application will run in multi-threaded CPU mode.")
print("==================================================")

In [ ]:
# ==========================================
# 7. MODEL PREPARATION & INITIALIZATION
# ==========================================
from app import fonts, transcriber

print("🔤 Initializing core Google fonts...")
fonts.ensure_fonts()

print("🎙️ Pre-warming local Whisper model weights...")
try:
    model = transcriber.load_model("auto")
    active_dev = transcriber.get_device()
    print(f"✅ Whisper model '{transcriber.MODEL_SIZE}' initialized and cached on {active_dev.upper()}.")
except Exception as e:
    print(f"⚠️ Model pre-warm notice: {e}")

In [ ]:
# ==========================================
# 8. START FASTAPI SERVER (BACKGROUND PROCESS)
# ==========================================
import time

LOG_FILE = REPO_DIR / "app_server.log"
if LOG_FILE.exists():
    LOG_FILE.unlink()

log_fp = open(LOG_FILE, "w", encoding="utf-8")
cmd = [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]

print("⚡ Starting ViralCut AI server background process...")
server_process = subprocess.Popen(cmd, stdout=log_fp, stderr=log_fp, cwd=str(REPO_DIR))

print(f"✅ Server process launched (PID: {server_process.pid}). Logging to '{LOG_FILE.name}'.")

In [ ]:
# ==========================================
# 9. SERVER HEALTH CHECK
# ==========================================
import requests

HEALTH_URL = "http://127.0.0.1:8000/health"
print("⏳ Waiting for server to initialize...")

server_ready = False
for attempt in range(1, 31):
    try:
        resp = requests.get(HEALTH_URL, timeout=2)
        if resp.status_code == 200:
            data = resp.json()
            print("==================================================")
            print(f"🎉 SERVER READY! Status: {data.get('status')} | Active Device: {data.get('device', 'unknown').upper()}")
            print("==================================================")
            server_ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not server_ready:
    print("❌ Server failed to respond to health check.")
    if LOG_FILE.exists():
        print(f"\n--- TAIL OF {LOG_FILE.name} ---")
        lines = LOG_FILE.read_text(encoding="utf-8").splitlines()[-25:]
        print("\n".join(lines))
    raise RuntimeError("ViralCut AI server failed to start.")

In [ ]:
# ==========================================
# 10. CLOUDFLARED PUBLIC TUNNEL SETUP
# ==========================================
import re

CLOUDFLARED_BIN = Path("/tmp/cloudflared")
if not CLOUDFLARED_BIN.exists():
    print("📦 Downloading Cloudflare Quick Tunnel binary...")
    dl_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    subprocess.run(["wget", "-q", "-O", str(CLOUDFLARED_BIN), dl_url], check=True)
    subprocess.run(["chmod", "+x", str(CLOUDFLARED_BIN)], check=True)

TUNNEL_LOG = REPO_DIR / "cloudflared.log"
if TUNNEL_LOG.exists():
    TUNNEL_LOG.unlink()

tunnel_fp = open(TUNNEL_LOG, "w", encoding="utf-8")
tunnel_cmd = [str(CLOUDFLARED_BIN), "tunnel", "--url", "http://127.0.0.1:8000"]

print("⚡ Launching Cloudflare Quick Tunnel...")
tunnel_process = subprocess.Popen(tunnel_cmd, stdout=tunnel_fp, stderr=tunnel_fp)

public_url = None
print("⏳ Resolving public URL from trycloudflare.com...")

for _ in range(30):
    time.sleep(1)
    if TUNNEL_LOG.exists():
        log_content = TUNNEL_LOG.read_text(encoding="utf-8")
        match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", log_content)
        if match:
            public_url = match.group(0)
            break

if public_url:
    print(f"🎉 Public Tunnel Established: {public_url}")
else:
    print("⚠️ Tunnel initialization pending. Checking log file...")
    if TUNNEL_LOG.exists():
        print("\n".join(TUNNEL_LOG.read_text(encoding="utf-8").splitlines()[-10:]))

In [ ]:
# ==========================================
# 11. DISPLAY ACCESSIBLE LAUNCH BOX
# ==========================================
from IPython.display import HTML, display

if public_url:
    html_content = f"""
    <div style="
        font-family: 'Outfit', 'Inter', -apple-system, sans-serif;
        background: linear-gradient(135deg, #0a0b10 0%, #14161e 100%);
        border: 1px solid #00C9FF;
        border-radius: 16px;
        padding: 24px;
        color: #f4f5f9;
        box-shadow: 0 12px 40px -10px rgba(0, 201, 255, 0.35);
        max-width: 680px;
        margin: 12px 0;
    ">
        <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 16px;">
            <div style="display: flex; align-items: center; gap: 12px;">
                <svg width="28" height="28" viewBox="0 0 24 24" fill="none" stroke="#00C9FF" stroke-width="2" stroke-linecap="round" stroke-linejoin="round">
                    <polygon points="13 2 3 14 12 14 11 22 21 10 12 10 13 2"></polygon>
                </svg>
                <div style="font-size: 20px; font-weight: 800; letter-spacing: -0.02em;">
                    ViralCut AI <span style="font-size: 13px; color: #A6FFFF; font-weight: 600;">by Kamran AI</span>
                </div>
            </div>
            <span style="
                background: rgba(0, 201, 255, 0.15);
                color: #A6FFFF;
                border: 1px solid rgba(0, 201, 255, 0.4);
                padding: 4px 12px;
                border-radius: 999px;
                font-size: 11px;
                font-weight: 700;
                text-transform: uppercase;
            ">Google Colab Ready</span>
        </div>
        
        <p style="font-size: 13.5px; color: #9aa1b2; margin-bottom: 20px; line-height: 1.5;">
            No setup required. No API tokens. Click the button below to launch the full ViralCut AI web interface directly in your browser.
        </p>
        
        <div style="margin-bottom: 20px;">
            <a href="{public_url}" target="_blank" style="
                display: inline-flex;
                align-items: center;
                gap: 10px;
                background: linear-gradient(135deg, #A6FFFF 0%, #00C9FF 55%, #008BFF 100%);
                color: #031422;
                font-weight: 800;
                font-size: 15px;
                padding: 14px 28px;
                border-radius: 10px;
                text-decoration: none;
                box-shadow: 0 8px 24px -6px rgba(0, 201, 255, 0.6);
            ">
                <span>🚀 Open ViralCut AI Dashboard</span>
                <svg width="16" height="16" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2.5">
                    <line x1="7" y1="17" x2="17" y2="7"></line>
                    <polyline points="7 7 17 7 17 17"></polyline>
                </svg>
            </a>
        </div>
        
        <div style="
            display: flex;
            gap: 16px;
            font-size: 12px;
            border-top: 1px solid rgba(255, 255, 255, 0.1);
            padding-top: 14px;
        ">
            <a href="{public_url}/docs" target="_blank" style="color: #00C9FF; text-decoration: none; font-weight: 600;">📚 OpenAPI Specs (/docs)</a>
            <span style="color: rgba(255, 255, 255, 0.2);">|</span>
            <a href="{public_url}/health" target="_blank" style="color: #00C9FF; text-decoration: none; font-weight: 600;">💓 Health Status (/health)</a>
        </div>
    </div>
    """
    display(HTML(html_content))
else:
    print("❌ Could not display launch box: Public URL unavailable.")

## 📖 Usage Guide

Follow these steps to generate short-form clips:

1. **Open Application UI**: Click the **🚀 Open ViralCut AI Dashboard** button above.
2. **Provide Source Video**:
   - **YouTube URL**: Paste any YouTube video link into the URL bar, OR
   - **Upload Local File**: Drag and drop an `.mp4`, `.mov`, or `.mkv` video file.
3. **Select Aspect Ratio**: Choose `9:16` (Vertical Shorts/Reels) or `16:9` (Landscape).
4. **Select Fit Mode**: Choose `crop` (Full vertical crop-fill) or `square` (1:1 Rounded Reel container on 9:16).
5. **Set Clip Count & Duration**: Choose how many clips to generate (e.g. 3) and optional target duration (e.g. 30s).
6. **Select Compute Engine**: Keep set to `Auto` (prefers GPU acceleration) or explicitly select `CUDA`.
7. **Select Creator Caption Style**: Pick from 18 built-in styles (`Hormozi Green`, `Beast Pop`, `Karaoke Yellow`, `Boxed TikTok`, `One-Word Punch`, etc.).
8. **Click Generate Shorts**: Watch real-time SSE progress updates across Download → Transcribe → Analyze → Render stages.
9. **Download Shorts**: Preview generated vertical clips directly in browser or download the ZIP archive.

> 💡 **Note on Colab Persistence**: Cloudflare Quick Tunnel URLs and Google Colab runtime storage are temporary. Download your generated clips before stopping or resetting the Colab runtime session.

In [ ]:
# ==========================================
# 13. APP & TUNNEL LOGS MONITOR
# ==========================================
print("=== APPLICATION SERVER LOGS (Tail 30 lines) ===")
if LOG_FILE.exists():
    print("\n".join(LOG_FILE.read_text(encoding="utf-8").splitlines()[-30:]))
else:
    print("No log file found.")

print("\n=== CLOUDFLARED TUNNEL LOGS (Tail 10 lines) ===")
if TUNNEL_LOG.exists():
    print("\n".join(TUNNEL_LOG.read_text(encoding="utf-8").splitlines()[-10:]))
else:
    print("No tunnel log file found.")

In [ ]:
# ==========================================
# 14. CLEAN APPLICATION SHUTDOWN
# ==========================================
print("🛑 Terminating application server and public tunnel processes...")

try:
    if 'server_process' in locals() and server_process.poll() is None:
        server_process.terminate()
        server_process.wait(timeout=3)
        print("✅ FastAPI server process stopped.")
except Exception as e:
    print(f"Notice stopping server: {e}")

try:
    if 'tunnel_process' in locals() and tunnel_process.poll() is None:
        tunnel_process.terminate()
        tunnel_process.wait(timeout=3)
        print("✅ Cloudflared tunnel process stopped.")
except Exception as e:
    print(f"Notice stopping tunnel: {e}")

print("🎉 Shutdown complete.")